In [1]:
import torch
import time
import keyboard

from hand_tracker_sig import CameraWrapper, HandTracker
from core import HumanSignalAutoencoder, OnlineKmeans, LiveDataset, AETrainer, CentroidLabeler, LabelGetter
from utils import HandVisualizer, StatsVisualizer

In [2]:
# architecture settings
human_signal_shape = (1, 21, 3)  # 1 hand * 21 tracking points * 3 coords
autoencoder_bottleneck = 42

# region classifier settings
okm_num_centroids      = 7
okm_retrain_iters      = 15
okm_convergence_atol   = 1e-6    # early-stops lloyd's when centroid shift falls below this
okm_min_cluster_divisor = 4      # centroid reinit threshold: starved when size < n/(k*divisor); lower = more aggressive reinit

# live dataset settings
queue_size_per_class = 200

# autoenc training settings
lr               = 0.0005
ae_batch_size    = 128
ae_steps_per_frame = 5           # gradient steps taken per camera frame
ae_weight_decay  = 0.01

# label settings
label_keys = ['w', 'a', 's', 'd']
num_labels = len(label_keys)

# label assignment confidence gate
label_margin_threshold = 0.75   # label rejected when nearest_dist / second_nearest_dist exceeds this; lower = stricter

# control keys
quit_key = 'q'

# hardware settings
camera_index         = 1
hand_landmarker_path = './hand_landmarker.task'

# mediapipe confidence thresholds
hand_detection_confidence = 0.5  # min score to initially detect a hand
hand_presence_confidence  = 0.5  # min score to keep tracking a detected hand
hand_tracking_confidence  = 0.5  # min score for landmark tracking

# torch settings
run_device = torch.device("cuda")

In [3]:
camera = CameraWrapper(camera_index)

hand_tracker = HandTracker(
    hand_landmarker_path,
    min_hand_detection_confidence=hand_detection_confidence,
    min_hand_presence_confidence=hand_presence_confidence,
    min_tracking_confidence=hand_tracking_confidence,
)

In [4]:
# open vis windows
true_visualizer = HandVisualizer('True Hand')
decoded_visualizer = HandVisualizer('Decoded Hand')
stats_visualizer = StatsVisualizer()

In [5]:
# initialize activeNCI core
autoenc = HumanSignalAutoencoder(human_signal_shape, autoencoder_bottleneck).to(run_device)
ae_trainer = AETrainer(autoenc, lr=lr, weight_decay=ae_weight_decay, device=run_device)

online_k_means = OnlineKmeans(okm_num_centroids, run_device,
                                  retrain_iters=okm_retrain_iters,
                                  convergence_atol=okm_convergence_atol,
                                  min_cluster_divisor=okm_min_cluster_divisor)

live_dataset = LiveDataset(okm_num_centroids, queue_size_per_class)

centroid_labeler = CentroidLabeler(okm_num_centroids, num_labels)
label_getter = LabelGetter(label_keys)

In [ ]:
start_time = time.time()
loss_val = float('nan')

while not keyboard.is_pressed(quit_key):

    img = camera()

    human_signal = hand_tracker(img, int((time.time() - start_time) * 1000)).to(run_device)
    if human_signal.shape != torch.Size(human_signal_shape):
        continue

    # encode
    with torch.no_grad():
        latent = autoenc.encode(human_signal.unsqueeze(0)).squeeze(0)

    # classify
    class_idx, dists = online_k_means.classify(latent)
    if class_idx is None:
        class_idx = 0

    # get ground truth label from held key
    label = label_getter.get_label()
    label_key_pressed = label is not None
    if label is not None and dists is not None:
        sorted_dists = dists.sort().values
        margin = sorted_dists[0] / (sorted_dists[1] + 1e-8)
        if margin >= label_margin_threshold:
            label = None

    # store datapoint
    live_dataset.apply_datapoint(latent, human_signal, class_idx, label=label)

    # retrain autoencoder
    for _ in range(ae_steps_per_frame):
        batch = live_dataset.get_random_human_sigs(ae_batch_size)
        if batch is not None:
            loss_val = ae_trainer.train_step(batch)

    # retrain OKM
    online_k_means.retrain(live_dataset.get_random_latents(live_dataset.total_size()))

    # recount labels when user is actively labeling
    if label_key_pressed:
        centroid_labeler.update(live_dataset)

    # resolve predicted key for this frame
    predicted_key_idx = centroid_labeler.predict(class_idx)
    predicted_key = label_keys[predicted_key_idx] if predicted_key_idx is not None else None

    # add custom action here

    # visualise
    true_visualizer.update(human_signal.detach().cpu())
    with torch.no_grad():
        decoded = autoenc(human_signal.unsqueeze(0)).squeeze(0)
    decoded_visualizer.update(decoded.detach().cpu())

    dist_val = dists[class_idx].item() if dists is not None else float('nan')
    label_display = ' | '.join(
        f"C{i}->{label_keys[p] if (p := centroid_labeler.predict(i)) is not None else '?'}"
        for i in range(okm_num_centroids)
    )
    stats_visualizer.update([
        f"AE loss: {loss_val:.6f} | batch: {ae_batch_size}",
        f"dataset queues: {live_dataset.get_queue_sizes()}",
        f"centroid: C{class_idx} (dist: {dist_val:.4f})",
        f"predicted: {predicted_key} | labels: {label_display}",
    ])